# 0825_peace_013_type_expert_fold_class_soft_voting

005와 007을 pooled Platt 보정 후 soft voting으로 결합하고, walk-forward에서만 alpha를 고정한 뒤 retrospective Test를 한 번만 평가한다.

In [1]:
import gc
import hashlib
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_peace_013_type_expert_fold_class_soft_voting"
RANDOM_STATE = 42
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"
DECISION_THRESHOLD = 0.5
MIN_RECALL = 0.99
TRAIN_END_FRACTION = 0.70
VALIDATION_END_FRACTION = 0.80
PLATT_SPLIT_FRACTION = 0.50
PROBABILITY_EPS = 1e-6
ALPHAS = [0.0, 0.25, 0.50, 0.75, 1.0]
ENSEMBLE_CHECKPOINTS = [0.30, 0.40, 0.50, 0.70]
FOLD_MEMBER_CHECKPOINTS = {
    "fold_1": [0.30],
    "fold_2": [0.30, 0.40],
    "fold_3": [0.30, 0.40, 0.50],
}
FINAL_MEMBER_CHECKPOINTS = ENSEMBLE_CHECKPOINTS.copy()
WALK_FORWARD_SPECS = [
    {
        "fold": "fold_1",
        "train_end": 0.30,
        "calibration_start": 0.30,
        "calibration_end": 0.40,
        "evaluation_start": 0.40,
        "evaluation_end": 0.50,
    },
    {
        "fold": "fold_2",
        "train_end": 0.40,
        "calibration_start": 0.40,
        "calibration_end": 0.50,
        "evaluation_start": 0.50,
        "evaluation_end": 0.60,
    },
    {
        "fold": "fold_3",
        "train_end": 0.50,
        "calibration_start": 0.50,
        "calibration_end": 0.60,
        "evaluation_start": 0.60,
        "evaluation_end": 0.70,
    },
]
XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "max_delta_step": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}
MODEL_LABELS = {
    "005": "005 expanding checkpoint ensemble",
    "007": "007 single type-expert + scale_pos_weight",
}

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


def compute_scale_pos_weight(y: pd.Series) -> float:
    positive = int(y.sum())
    negative = int(len(y) - positive)
    if positive == 0 or negative == 0:
        raise ValueError("scale_pos_weight는 양성과 음성이 모두 있는 Train에서만 계산할 수 있습니다.")
    return negative / positive


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("AGENTS.md가 있는 저장소 루트를 찾지 못했습니다.")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_data_pair(repo_root: Path) -> tuple[Path, Path]:
    candidates = [
        repo_root / "data" / "raw",
        repo_root.parent,
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    checked = set()
    for directory in candidates:
        resolved = directory.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        data_path = resolved / "dataset.csv"
        mapping_path = resolved / "mapping.json"
        if data_path.exists() and mapping_path.exists():
            return data_path, mapping_path
    raise FileNotFoundError("dataset.csv와 mapping.json 쌍을 찾지 못했습니다.")


def to_series(index, values):
    return pd.Series(np.asarray(values, dtype=np.float64), index=index, dtype="float64")


def evaluate_predictions(y_true, prediction, probability):
    y_true = np.asarray(y_true, dtype=np.int8)
    prediction = np.asarray(prediction, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2
    has_positive = (tp + fn) > 0
    return {
        "rows": len(y_true),
        "positive_samples": int(y_true.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "false_call_reduction": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "roc_auc": roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, probability) if has_both_classes else np.nan,
    }


def evaluate_probabilities(y_true, probability, threshold=DECISION_THRESHOLD):
    probability = np.asarray(probability, dtype=np.float64)
    prediction = (probability >= threshold).astype(np.int8)
    return evaluate_predictions(y_true, prediction, probability)


def select_threshold(y_true, probability, min_recall=MIN_RECALL):
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    if np.unique(y_true).size != 2:
        raise ValueError("임계값 선택에는 positive와 negative가 모두 필요합니다.")

    order = np.argsort(-probability, kind="stable")
    sorted_probability = probability[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target == 1)
    cumulative_fp = np.cumsum(sorted_target == 0)
    group_ends = np.flatnonzero(np.r_[sorted_probability[:-1] != sorted_probability[1:], True])

    thresholds = sorted_probability[group_ends]
    tp = cumulative_tp[group_ends]
    fp = cumulative_fp[group_ends]
    total_positive = int((y_true == 1).sum())
    total_negative = int((y_true == 0).sum())
    recall = tp / total_positive
    false_call_reduction = 1.0 - (fp / total_negative)
    feasible = np.flatnonzero(recall >= min_recall)
    if feasible.size == 0:
        raise RuntimeError(f"Recall {min_recall:.2%} 조건을 만족하는 threshold가 없습니다.")

    best_local = np.lexsort((thresholds[feasible], recall[feasible], false_call_reduction[feasible]))[-1]
    best = feasible[best_local]
    selected_threshold = float(thresholds[best])
    metrics = evaluate_probabilities(y_true, probability, selected_threshold)
    return {"threshold": selected_threshold, "min_recall": min_recall, **metrics}


def make_preprocessor(feature_columns, meta_columns):
    categorical = [column for column in meta_columns if column in feature_columns]
    continuous = [column for column in feature_columns if column not in categorical]
    return ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore", dtype=np.float32), categorical),
            ("continuous", "passthrough", continuous),
        ],
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )


def safe_logit(probability, eps=PROBABILITY_EPS):
    probability = np.asarray(probability, dtype=np.float64)
    clipped = np.clip(probability, eps, 1.0 - eps)
    return np.log(clipped / (1.0 - clipped))


def fit_platt_calibrator(y_true, raw_probability, model_key, stage_name):
    y_true = np.asarray(y_true, dtype=np.int8)
    X = safe_logit(raw_probability).reshape(-1, 1)
    assert np.unique(y_true).size == 2
    calibrator = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
    calibrator.fit(X, y_true)
    row = {
        "stage": stage_name,
        "model_key": model_key,
        "model_label": MODEL_LABELS[model_key],
        "rows": int(len(y_true)),
        "positive_samples": int(y_true.sum()),
        "negative_samples": int(len(y_true) - y_true.sum()),
        "coefficient": float(calibrator.coef_[0, 0]),
        "intercept": float(calibrator.intercept_[0]),
        "raw_probability_min": float(np.min(raw_probability)),
        "raw_probability_max": float(np.max(raw_probability)),
    }
    logger.info(
        "platt_fit_done stage=%s model=%s rows=%d positive=%d coef=%.8f intercept=%.8f",
        stage_name,
        model_key,
        row["rows"],
        row["positive_samples"],
        row["coefficient"],
        row["intercept"],
    )
    return calibrator, row


def apply_platt_calibrator(calibrator, raw_probability):
    X = safe_logit(raw_probability).reshape(-1, 1)
    return calibrator.predict_proba(X)[:, 1]


def split_by_timestamp_groups(frame, stage_name, fraction=PLATT_SPLIT_FRACTION):
    if frame.empty:
        raise ValueError(f"{stage_name}: 빈 frame은 분할할 수 없습니다.")
    ordered = frame.sort_values([TIME_COLUMN, RECORD_ID], kind="stable")
    group_counts = ordered.groupby(TIME_COLUMN, sort=True).size()
    if len(group_counts) < 2:
        raise ValueError(f"{stage_name}: timestamp group이 2개 미만입니다.")
    cumulative_rows = group_counts.cumsum().to_numpy()
    target_rows = len(ordered) * fraction
    candidate_positions = np.arange(len(group_counts) - 1)
    ordered_positions = candidate_positions[np.argsort(np.abs(cumulative_rows[:-1] - target_rows), kind="stable")]
    for pos in ordered_positions:
        cut_timestamp = group_counts.index[pos]
        earlier = ordered.loc[ordered[TIME_COLUMN] <= cut_timestamp]
        later = ordered.loc[ordered[TIME_COLUMN] > cut_timestamp]
        if earlier.empty or later.empty:
            continue
        if earlier[TARGET].nunique() != 2 or later[TARGET].nunique() != 2:
            continue
        assert earlier[TIME_COLUMN].max() < later[TIME_COLUMN].min()
        assert set(earlier[TIME_COLUMN]).isdisjoint(set(later[TIME_COLUMN]))
        rows = [
            {
                "stage": stage_name,
                "segment": "platt_fit",
                "rows": len(earlier),
                "positive_samples": int(earlier[TARGET].sum()),
                "positive_rate_pct": earlier[TARGET].mean() * 100,
                "timestamp_groups": earlier[TIME_COLUMN].nunique(),
                "start_time": earlier[TIME_COLUMN].min(),
                "end_time": earlier[TIME_COLUMN].max(),
            },
            {
                "stage": stage_name,
                "segment": "threshold_selection",
                "rows": len(later),
                "positive_samples": int(later[TARGET].sum()),
                "positive_rate_pct": later[TARGET].mean() * 100,
                "timestamp_groups": later[TIME_COLUMN].nunique(),
                "start_time": later[TIME_COLUMN].min(),
                "end_time": later[TIME_COLUMN].max(),
            },
        ]
        logger.info(
            "temporal_split_done stage=%s cut_timestamp=%s earlier_rows=%d later_rows=%d earlier_positive=%d later_positive=%d",
            stage_name,
            cut_timestamp,
            len(earlier),
            len(later),
            int(earlier[TARGET].sum()),
            int(later[TARGET].sum()),
        )
        return earlier, later, rows
    raise RuntimeError(f"{stage_name}: 양쪽 모두 두 클래스를 포함하는 timestamp 분할을 찾지 못했습니다.")


def predict_type_experts(train_frame, target_frames, use_class_weight, stage_name):
    prediction_map = {name: pd.Series(np.nan, index=frame.index, dtype="float64") for name, frame in target_frames.items()}
    training_rows = []
    for inspection_type in inspection_types:
        feature_columns = feature_columns_by_type[inspection_type]
        type_train = train_frame.loc[train_frame[TYPE_COLUMN] == inspection_type]
        y_train = type_train[TARGET].astype("int8")
        assert len(type_train) > 0
        assert y_train.nunique() == 2
        scale_pos_weight = compute_scale_pos_weight(y_train) if use_class_weight else 1.0
        preprocessor = make_preprocessor(feature_columns, meta_columns)
        X_train = preprocessor.fit_transform(type_train[feature_columns])
        model_params = XGB_PARAMS.copy()
        if use_class_weight:
            model_params["scale_pos_weight"] = scale_pos_weight
        model = XGBClassifier(**model_params)
        model.fit(X_train, y_train, verbose=False)
        for target_name, frame in target_frames.items():
            type_target = frame.loc[frame[TYPE_COLUMN] == inspection_type]
            X_target = preprocessor.transform(type_target[feature_columns])
            prediction_map[target_name].loc[type_target.index] = model.predict_proba(X_target)[:, 1]
            del X_target
        training_rows.append(
            {
                "stage": stage_name,
                "model_family": "007" if use_class_weight else "type_expert",
                "inspection_type": inspection_type,
                "train_rows": len(type_train),
                "train_positive": int(y_train.sum()),
                "raw_features": len(feature_columns),
                "encoded_features": X_train.shape[1],
                "scale_pos_weight": float(scale_pos_weight),
            }
        )
        logger.info(
            "fit_done stage=%s type=%d use_class_weight=%s train_rows=%d positives=%d",
            stage_name,
            inspection_type,
            use_class_weight,
            len(type_train),
            int(y_train.sum()),
        )
        del preprocessor, model, X_train
        gc.collect()
    for target_name, series in prediction_map.items():
        assert series.notna().all(), (stage_name, target_name)
    return prediction_map, training_rows


def build_005_walk_predictions():
    prediction_targets = {}
    for spec in WALK_FORWARD_SPECS:
        fold_name = spec["fold"]
        prediction_targets[f"{fold_name}_calibration"] = walk_forward_segments[fold_name]["calibration"]
        prediction_targets[f"{fold_name}_evaluation"] = walk_forward_segments[fold_name]["evaluation"]
    checkpoint_predictions = {
        checkpoint: {
            target_name: pd.Series(np.nan, index=frame.index, dtype="float64")
            for target_name, frame in prediction_targets.items()
            if frame[TIME_COLUMN].min() > walk_forward_boundaries[checkpoint]
        }
        for checkpoint in ENSEMBLE_CHECKPOINTS
    }
    training_rows = []
    for checkpoint in ENSEMBLE_CHECKPOINTS:
        checkpoint_train = raw_df.loc[raw_df[TIME_COLUMN] <= walk_forward_boundaries[checkpoint]]
        target_frames = {name: prediction_targets[name] for name in checkpoint_predictions[checkpoint]}
        prediction_map, rows = predict_type_experts(
            checkpoint_train,
            target_frames,
            use_class_weight=False,
            stage_name=f"walk_checkpoint_{checkpoint:.2f}",
        )
        checkpoint_predictions[checkpoint] = prediction_map
        training_rows.extend(rows)
    fold_predictions = {}
    for fold_name, members in FOLD_MEMBER_CHECKPOINTS.items():
        calibration_name = f"{fold_name}_calibration"
        evaluation_name = f"{fold_name}_evaluation"
        fold_predictions[fold_name] = {
            "calibration": to_series(
                walk_forward_segments[fold_name]["calibration"].index,
                np.mean(np.vstack([checkpoint_predictions[checkpoint][calibration_name].to_numpy() for checkpoint in members]), axis=0),
            ),
            "evaluation": to_series(
                walk_forward_segments[fold_name]["evaluation"].index,
                np.mean(np.vstack([checkpoint_predictions[checkpoint][evaluation_name].to_numpy() for checkpoint in members]), axis=0),
            ),
        }
    return fold_predictions, training_rows


def build_007_walk_predictions():
    fold_predictions = {}
    training_rows = []
    for spec in WALK_FORWARD_SPECS:
        fold_name = spec["fold"]
        segments = walk_forward_segments[fold_name]
        prediction_map, rows = predict_type_experts(
            segments["train"],
            {"calibration": segments["calibration"], "evaluation": segments["evaluation"]},
            use_class_weight=True,
            stage_name=f"walk_{fold_name}",
        )
        fold_predictions[fold_name] = prediction_map
        training_rows.extend(rows)
    return fold_predictions, training_rows


def build_005_final_predictions():
    prediction_targets = {"validation": validation_df, "test": test_df}
    checkpoint_predictions = {}
    training_rows = []
    for checkpoint in FINAL_MEMBER_CHECKPOINTS:
        checkpoint_train = raw_df.loc[raw_df[TIME_COLUMN] <= walk_forward_boundaries[checkpoint]]
        prediction_map, rows = predict_type_experts(
            checkpoint_train,
            prediction_targets,
            use_class_weight=False,
            stage_name=f"final_checkpoint_{checkpoint:.2f}",
        )
        checkpoint_predictions[checkpoint] = prediction_map
        training_rows.extend(rows)
    validation_probability = to_series(
        validation_df.index,
        np.mean(np.vstack([checkpoint_predictions[checkpoint]["validation"].to_numpy() for checkpoint in FINAL_MEMBER_CHECKPOINTS]), axis=0),
    )
    test_probability = to_series(
        test_df.index,
        np.mean(np.vstack([checkpoint_predictions[checkpoint]["test"].to_numpy() for checkpoint in FINAL_MEMBER_CHECKPOINTS]), axis=0),
    )
    return {"validation": validation_probability, "test": test_probability}, training_rows


def build_007_final_predictions():
    return predict_type_experts(
        train_df,
        {"validation": validation_df, "test": test_df},
        use_class_weight=True,
        stage_name="final_train70",
    )


def blend_probabilities(prob_005, prob_007, alpha):
    return alpha * np.asarray(prob_005, dtype=np.float64) + (1.0 - alpha) * np.asarray(prob_007, dtype=np.float64)


def pct(value: float) -> str:
    if pd.isna(value):
        return "nan"
    return f"{value * 100:.2f}%"


def md_table(df: pd.DataFrame, float_formats=None) -> str:
    float_formats = float_formats or {}
    formatted = df.copy()
    for column, formatter in float_formats.items():
        if column in formatted.columns:
            formatted[column] = formatted[column].map(formatter)
    return formatted.to_markdown(index=False)


def alpha_label(alpha, selected_alpha):
    if np.isclose(alpha, selected_alpha):
        return f"alpha={alpha:.2f} (walk-forward selected)"
    if np.isclose(alpha, 0.0):
        return "alpha=0.00 (007 only diagnostic)"
    if np.isclose(alpha, 1.0):
        return "alpha=1.00 (005 only diagnostic)"
    return f"alpha={alpha:.2f}"


def alpha_role(alpha, selected_alpha):
    if np.isclose(alpha, selected_alpha):
        return "selected_walk_forward"
    if np.isclose(alpha, 0.0) or np.isclose(alpha, 1.0):
        return "diagnostic_endpoint"
    return "candidate_not_selected"


def endpoint_sort_key(alpha, selected_alpha):
    if np.isclose(alpha, 0.0):
        return 0
    if np.isclose(alpha, selected_alpha):
        return 1
    if np.isclose(alpha, 1.0):
        return 2
    return 3


REPO_ROOT = find_repo_root()
DATA_PATH, MAPPING_PATH = find_data_pair(REPO_ROOT)
REPORT_PATH = REPO_ROOT / "docs" / "experiments" / f"{EXPERIMENT_ID}.md"
LOG_DIR = REPO_ROOT / "docs" / "peace"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"{EXPERIMENT_ID}.log"

logger = logging.getLogger(EXPERIMENT_ID)
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.propagate = False

DATA_SHA256_BEFORE = sha256_file(DATA_PATH)
MAPPING_SHA256_BEFORE = sha256_file(MAPPING_PATH)
logger.info("experiment=%s", EXPERIMENT_ID)
logger.info("data_file=%s sha256=%s", DATA_PATH.name, DATA_SHA256_BEFORE)
logger.info("mapping_file=%s sha256=%s", MAPPING_PATH.name, MAPPING_SHA256_BEFORE)
logger.info(
    "versions python=%s pandas=%s sklearn=%s xgboost=%s",
    sys.version.split()[0],
    pd.__version__,
    sklearn.__version__,
    xgboost.__version__,
)
logger.info("alphas=%s min_recall=%.2f", ALPHAS, MIN_RECALL)
print("log saved to:", LOG_PATH.relative_to(REPO_ROOT))

raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:") or source_index_column == "":
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

with MAPPING_PATH.open(encoding="utf-8") as stream:
    feature_mapping = json.load(stream)

required_columns = {RECORD_ID, TIME_COLUMN, TYPE_COLUMN, TARGET}
missing_required = required_columns - set(raw_df.columns)
assert not missing_required, f"필수 컬럼 누락: {sorted(missing_required)}"
assert len(raw_df) == 440_274
assert raw_df[RECORD_ID].is_unique
assert set(raw_df[TARGET].unique()) == {0, 1}
assert raw_df[TARGET].value_counts().to_dict() == {0: 435_652, 1: 4_622}
assert set(raw_df[TYPE_COLUMN].unique()) == {0, 1, 2, 3, 4}
assert set(feature_mapping) == {"0", "1", "2", "3", "4"}

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)
inspection_columns = [column for column in raw_df.columns if column.startswith("inspection_feat")]
mapped_union = set().union(*(set(columns) for columns in feature_mapping.values()))
assert len(inspection_columns) == 70
assert len(mapped_union) == 65
assert mapped_union <= set(inspection_columns)

numeric_inputs = raw_df.select_dtypes(include=[np.number]).drop(columns=[TARGET, RECORD_ID])
assert np.isfinite(numeric_inputs.to_numpy()).all()

inspection_types = sorted(raw_df[TYPE_COLUMN].unique().tolist())
meta_columns = [column for column in raw_df.columns if column.startswith("meta_feat")]
feature_columns_by_type = {}
feature_rows = []
for inspection_type in inspection_types:
    mapped_columns = feature_mapping[str(inspection_type)]
    assert len(mapped_columns) == len(set(mapped_columns))
    assert set(mapped_columns) <= set(raw_df.columns)
    selected_columns = meta_columns + mapped_columns
    feature_columns_by_type[inspection_type] = selected_columns
    feature_rows.append(
        {
            "inspection_type": inspection_type,
            "meta_features": len(meta_columns),
            "mapped_inspection_features": len(mapped_columns),
            "raw_features_used": len(selected_columns),
        }
    )
feature_summary = pd.DataFrame(feature_rows).set_index("inspection_type")

data_summary = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "false_call_0": int((raw_df[TARGET] == 0).sum()),
        "real_defect_1": int((raw_df[TARGET] == 1).sum()),
        "real_defect_rate_pct": raw_df[TARGET].mean() * 100,
        "inspection_types": raw_df[TYPE_COLUMN].nunique(),
        "inspection_features": len(inspection_columns),
        "mapped_feature_union": len(mapped_union),
        "timestamp_start": raw_df[TIME_COLUMN].min(),
        "timestamp_end": raw_df[TIME_COLUMN].max(),
    },
    name="raw_data",
)
display(data_summary)
display(feature_summary)
logger.info("data_verified rows=%d columns=%d class_0=%d class_1=%d", len(raw_df), raw_df.shape[1], int((raw_df[TARGET] == 0).sum()), int((raw_df[TARGET] == 1).sum()))

timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index


def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side="left"))
    return timestamp_index[position]


train_end_time = boundary_at(TRAIN_END_FRACTION)
validation_end_time = boundary_at(VALIDATION_END_FRACTION)
train_mask = raw_df[TIME_COLUMN] <= train_end_time
validation_mask = (raw_df[TIME_COLUMN] > train_end_time) & (raw_df[TIME_COLUMN] <= validation_end_time)
test_mask = raw_df[TIME_COLUMN] > validation_end_time

train_df = raw_df.loc[train_mask]
validation_df = raw_df.loc[validation_mask]
test_df = raw_df.loc[test_mask]
assert train_df[TIME_COLUMN].max() < validation_df[TIME_COLUMN].min()
assert validation_df[TIME_COLUMN].max() < test_df[TIME_COLUMN].min()
assert int(train_mask.sum() + validation_mask.sum() + test_mask.sum()) == len(raw_df)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "positive_samples": int(train_df[TARGET].sum()),
            "positive_rate_pct": train_df[TARGET].mean() * 100,
            "timestamp_groups": train_df[TIME_COLUMN].nunique(),
            "start_time": train_df[TIME_COLUMN].min(),
            "end_time": train_df[TIME_COLUMN].max(),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "positive_samples": int(validation_df[TARGET].sum()),
            "positive_rate_pct": validation_df[TARGET].mean() * 100,
            "timestamp_groups": validation_df[TIME_COLUMN].nunique(),
            "start_time": validation_df[TIME_COLUMN].min(),
            "end_time": validation_df[TIME_COLUMN].max(),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "positive_samples": int(test_df[TARGET].sum()),
            "positive_rate_pct": test_df[TARGET].mean() * 100,
            "timestamp_groups": test_df[TIME_COLUMN].nunique(),
            "start_time": test_df[TIME_COLUMN].min(),
            "end_time": test_df[TIME_COLUMN].max(),
        },
    ]
).set_index("split")
display(split_summary)
logger.info("split_summary=%s", split_summary.reset_index().to_dict(orient="records"))

walk_forward_boundaries = {fraction: boundary_at(fraction) for fraction in [0.30, 0.40, 0.50, 0.60, 0.70]}
walk_forward_segments = {}
walk_forward_split_rows = []
for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    train_end = walk_forward_boundaries[spec["train_end"]]
    calibration_start = walk_forward_boundaries[spec["calibration_start"]]
    calibration_end = walk_forward_boundaries[spec["calibration_end"]]
    evaluation_start = walk_forward_boundaries[spec["evaluation_start"]]
    evaluation_end = walk_forward_boundaries[spec["evaluation_end"]]
    segments = {
        "train": raw_df.loc[raw_df[TIME_COLUMN] <= train_end],
        "calibration": raw_df.loc[(raw_df[TIME_COLUMN] > calibration_start) & (raw_df[TIME_COLUMN] <= calibration_end)],
        "evaluation": raw_df.loc[(raw_df[TIME_COLUMN] > evaluation_start) & (raw_df[TIME_COLUMN] <= evaluation_end)],
    }
    assert segments["train"][TIME_COLUMN].max() < segments["calibration"][TIME_COLUMN].min()
    assert segments["calibration"][TIME_COLUMN].max() < segments["evaluation"][TIME_COLUMN].min()
    assert set(segments["train"][TIME_COLUMN]).isdisjoint(segments["calibration"][TIME_COLUMN])
    assert set(segments["calibration"][TIME_COLUMN]).isdisjoint(segments["evaluation"][TIME_COLUMN])
    walk_forward_segments[fold_name] = segments
    for segment_name, frame in segments.items():
        walk_forward_split_rows.append(
            {
                "fold": fold_name,
                "segment": segment_name,
                "rows": len(frame),
                "positive_samples": int(frame[TARGET].sum()),
                "positive_rate_pct": frame[TARGET].mean() * 100,
                "timestamp_groups": frame[TIME_COLUMN].nunique(),
                "start_time": frame[TIME_COLUMN].min(),
                "end_time": frame[TIME_COLUMN].max(),
            }
        )
walk_forward_split_summary = pd.DataFrame(walk_forward_split_rows).set_index(["fold", "segment"])
display(walk_forward_split_summary)
logger.info("walk_forward_split_summary=%s", walk_forward_split_summary.reset_index().to_dict(orient="records"))

walk_005_predictions, walk_005_training = build_005_walk_predictions()
walk_007_predictions, walk_007_training = build_007_walk_predictions()
final_005_predictions, final_005_training = build_005_final_predictions()
final_007_predictions, final_007_training = build_007_final_predictions()
training_history = pd.DataFrame(walk_005_training + walk_007_training + final_005_training + final_007_training)
display(training_history.head())

walk_temporal_split_rows = []
platt_rows = []
walk_alpha_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    calibration_frame = walk_forward_segments[fold_name]["calibration"]
    evaluation_frame = walk_forward_segments[fold_name]["evaluation"]
    platt_fit_frame, threshold_frame, temporal_rows = split_by_timestamp_groups(calibration_frame, fold_name)
    walk_temporal_split_rows.extend(temporal_rows)

    threshold_probabilities = {}
    evaluation_probabilities = {}
    for model_key, raw_store in [("005", walk_005_predictions), ("007", walk_007_predictions)]:
        platt_fit_probability = raw_store[fold_name]["calibration"].loc[platt_fit_frame.index].to_numpy()
        calibrator, platt_row = fit_platt_calibrator(
            platt_fit_frame[TARGET].to_numpy(),
            platt_fit_probability,
            model_key=model_key,
            stage_name=fold_name,
        )
        platt_rows.append(platt_row)
        threshold_probabilities[model_key] = apply_platt_calibrator(
            calibrator,
            raw_store[fold_name]["calibration"].loc[threshold_frame.index].to_numpy(),
        )
        evaluation_probabilities[model_key] = apply_platt_calibrator(
            calibrator,
            raw_store[fold_name]["evaluation"].to_numpy(),
        )

    for alpha in ALPHAS:
        threshold_probability = blend_probabilities(threshold_probabilities["005"], threshold_probabilities["007"], alpha)
        evaluation_probability = blend_probabilities(evaluation_probabilities["005"], evaluation_probabilities["007"], alpha)
        selection = select_threshold(threshold_frame[TARGET].to_numpy(), threshold_probability, min_recall=MIN_RECALL)
        future_metrics = evaluate_probabilities(evaluation_frame[TARGET].to_numpy(), evaluation_probability, threshold=selection["threshold"])
        walk_alpha_rows.append(
            {
                "fold": fold_name,
                "alpha": alpha,
                "weight_005": alpha,
                "weight_007": 1.0 - alpha,
                "threshold_selected": float(selection["threshold"]),
                "threshold_recall": float(selection["recall"]),
                "threshold_fp": int(selection["fp"]),
                "threshold_fcr": float(selection["false_call_reduction"]),
                "future_recall": float(future_metrics["recall"]),
                "future_fp": int(future_metrics["fp"]),
                "future_fn": int(future_metrics["fn"]),
                "future_tp": int(future_metrics["tp"]),
                "future_tn": int(future_metrics["tn"]),
                "future_fcr": float(future_metrics["false_call_reduction"]),
                "future_precision": float(future_metrics["precision"]),
                "future_pr_auc": float(future_metrics["pr_auc"]),
            }
        )
        logger.info(
            "walk_alpha_done fold=%s alpha=%.2f threshold=%.8f future_recall=%.6f future_fp=%d future_fcr=%.6f",
            fold_name,
            alpha,
            selection["threshold"],
            future_metrics["recall"],
            future_metrics["fp"],
            future_metrics["false_call_reduction"],
        )

walk_temporal_split_summary = pd.DataFrame(walk_temporal_split_rows)
platt_summary = pd.DataFrame(platt_rows)
walk_alpha_results = pd.DataFrame(walk_alpha_rows).sort_values(["alpha", "fold"]).reset_index(drop=True)
walk_alpha_summary = (
    walk_alpha_results.groupby("alpha", sort=True)
    .agg(
        folds=("fold", "nunique"),
        recall_target_hit_folds=("future_recall", lambda values: int((values >= MIN_RECALL).sum())),
        mean_future_recall=("future_recall", "mean"),
        min_future_recall=("future_recall", "min"),
        mean_future_fp=("future_fp", "mean"),
        total_future_fp=("future_fp", "sum"),
        mean_future_fcr=("future_fcr", "mean"),
        min_future_fcr=("future_fcr", "min"),
        mean_future_pr_auc=("future_pr_auc", "mean"),
    )
    .reset_index()
    .sort_values(
        ["recall_target_hit_folds", "min_future_recall", "total_future_fp", "mean_future_pr_auc", "alpha"],
        ascending=[False, False, True, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)
selected_alpha = float(walk_alpha_summary.iloc[0]["alpha"])
display(walk_temporal_split_summary)
display(platt_summary)
display(walk_alpha_results)
display(walk_alpha_summary)
logger.info("selected_alpha=%.2f summary=%s", selected_alpha, walk_alpha_summary.iloc[0].to_dict())

final_platt_fit_frame, final_threshold_frame, final_temporal_rows = split_by_timestamp_groups(validation_df, "final_validation")
final_temporal_split_summary = pd.DataFrame(final_temporal_rows)
final_platt_rows = []
final_threshold_results = []

final_threshold_probabilities = {}
final_test_probabilities = {}
for model_key, prediction_map in [("005", final_005_predictions), ("007", final_007_predictions)]:
    calibrator, platt_row = fit_platt_calibrator(
        final_platt_fit_frame[TARGET].to_numpy(),
        prediction_map["validation"].loc[final_platt_fit_frame.index].to_numpy(),
        model_key=model_key,
        stage_name="final_validation",
    )
    final_platt_rows.append(platt_row)
    final_threshold_probabilities[model_key] = apply_platt_calibrator(
        calibrator,
        prediction_map["validation"].loc[final_threshold_frame.index].to_numpy(),
    )
    final_test_probabilities[model_key] = apply_platt_calibrator(calibrator, prediction_map["test"].to_numpy())

final_platt_summary = pd.DataFrame(final_platt_rows)
for alpha in ALPHAS:
    threshold_probability = blend_probabilities(final_threshold_probabilities["005"], final_threshold_probabilities["007"], alpha)
    test_probability = blend_probabilities(final_test_probabilities["005"], final_test_probabilities["007"], alpha)
    selection = select_threshold(final_threshold_frame[TARGET].to_numpy(), threshold_probability, min_recall=MIN_RECALL)
    test_metrics = evaluate_probabilities(test_df[TARGET].to_numpy(), test_probability, threshold=selection["threshold"])
    final_threshold_results.append(
        {
            "alpha": alpha,
            "weight_005": alpha,
            "weight_007": 1.0 - alpha,
            "role": alpha_role(alpha, selected_alpha),
            "validation_threshold": float(selection["threshold"]),
            "validation_threshold_recall": float(selection["recall"]),
            "validation_threshold_fp": int(selection["fp"]),
            "validation_threshold_fcr": float(selection["false_call_reduction"]),
            "test_recall": float(test_metrics["recall"]),
            "test_fp": int(test_metrics["fp"]),
            "test_fcr": float(test_metrics["false_call_reduction"]),
            "test_fn": int(test_metrics["fn"]),
            "test_tp": int(test_metrics["tp"]),
            "test_tn": int(test_metrics["tn"]),
            "test_precision": float(test_metrics["precision"]),
            "test_pr_auc": float(test_metrics["pr_auc"]),
            "selection_note": "Test reused from prior experiments and this retrospective; alpha/threshold were fixed without using Test labels for model selection.",
        }
    )
    logger.info(
        "final_alpha_done alpha=%.2f threshold=%.8f test_recall=%.6f test_fp=%d test_fcr=%.6f role=%s",
        alpha,
        selection["threshold"],
        test_metrics["recall"],
        test_metrics["fp"],
        test_metrics["false_call_reduction"],
        alpha_role(alpha, selected_alpha),
    )

final_results = pd.DataFrame(final_threshold_results).sort_values("alpha").reset_index(drop=True)
selected_endpoints_df = final_results.loc[final_results["alpha"].isin([0.0, selected_alpha, 1.0])].copy()
selected_endpoints_df["label"] = selected_endpoints_df["alpha"].map(lambda value: alpha_label(value, selected_alpha))
selected_endpoints_df["sort_key"] = selected_endpoints_df["alpha"].map(lambda value: endpoint_sort_key(value, selected_alpha))
selected_endpoints_df = selected_endpoints_df.sort_values(["sort_key", "alpha"]).drop(columns=["sort_key"])
display(final_temporal_split_summary)
display(final_platt_summary)
display(final_results)
display(selected_endpoints_df[["label", "validation_threshold", "test_recall", "test_fp", "test_fcr", "test_fn", "test_tp", "test_pr_auc", "role"]])

DATA_SHA256_AFTER = sha256_file(DATA_PATH)
MAPPING_SHA256_AFTER = sha256_file(MAPPING_PATH)
assert DATA_SHA256_AFTER == DATA_SHA256_BEFORE
assert MAPPING_SHA256_AFTER == MAPPING_SHA256_BEFORE

verification = pd.Series(
    {
        "dataset_sha256_unchanged": True,
        "mapping_sha256_unchanged": True,
        "dataset_sha256": DATA_SHA256_BEFORE,
        "mapping_sha256": MAPPING_SHA256_BEFORE,
        "selected_alpha_from_walk_forward": selected_alpha,
        "walk_fold_count": int(walk_alpha_results["fold"].nunique()),
        "walk_candidate_count": int(len(walk_alpha_summary)),
        "final_candidate_count": int(len(final_results)),
        "log_file": f"docs/peace/{LOG_PATH.name}",
        "report_file": f"docs/experiments/{REPORT_PATH.name}",
    },
    name="verification",
)
display(verification)
logger.info("source_integrity=PASS")

walk_alpha_report = walk_alpha_results[
    [
        "fold",
        "alpha",
        "weight_005",
        "weight_007",
        "threshold_selected",
        "threshold_recall",
        "threshold_fp",
        "threshold_fcr",
        "future_recall",
        "future_fp",
        "future_fcr",
        "future_fn",
        "future_pr_auc",
    ]
]
platt_report = pd.concat([platt_summary, final_platt_summary], ignore_index=True)
selected_text = (
    "Selection hierarchy on walk-forward only: "
    "1) maximize Recall 99% hit folds, "
    "2) maximize minimum future Recall, "
    "3) minimize total future FP, "
    "4) maximize mean future PR-AUC, "
    "5) lower alpha only as deterministic tie-break."
)

selected_alpha_row = walk_alpha_summary.loc[np.isclose(walk_alpha_summary["alpha"], selected_alpha)].iloc[0]
alpha_zero_row = final_results.loc[np.isclose(final_results["alpha"], 0.0)].iloc[0]
alpha_one_row = final_results.loc[np.isclose(final_results["alpha"], 1.0)].iloc[0]
chosen_row = final_results.loc[np.isclose(final_results["alpha"], selected_alpha)].iloc[0]
conclusion_lines = [
    f"- Walk-forward selected alpha is {selected_alpha:.2f} (005 weight {selected_alpha:.2f}, 007 weight {1.0 - selected_alpha:.2f}). It hit Recall 99% in {int(selected_alpha_row['recall_target_hit_folds'])}/3 folds, minimum future Recall {pct(selected_alpha_row['min_future_recall'])}, and total future FP {int(selected_alpha_row['total_future_fp']):,}.",
    f"- Final retrospective with frozen alpha {selected_alpha:.2f} reached Test Recall {pct(chosen_row['test_recall'])}, FP {int(chosen_row['test_fp']):,}, FCR {pct(chosen_row['test_fcr'])}, FN {int(chosen_row['test_fn'])}, TP {int(chosen_row['test_tp'])}, PR-AUC {chosen_row['test_pr_auc']:.6f}.",
    f"- Diagnostic endpoint alpha=1.00 (005 only) produced Test Recall {pct(alpha_one_row['test_recall'])}, FP {int(alpha_one_row['test_fp']):,}, FCR {pct(alpha_one_row['test_fcr'])}.",
    f"- Diagnostic endpoint alpha=0.00 (007 only) produced Test Recall {pct(alpha_zero_row['test_recall'])}, FP {int(alpha_zero_row['test_fp']):,}, FCR {pct(alpha_zero_row['test_fcr'])}.",
    "- The retrospective Test was reused from prior experiments and was not used to choose alpha; alpha was fixed strictly from walk-forward summary.",
]

report_text = "\n".join(
    [
        f"# {EXPERIMENT_ID}",
        "",
        "## 연결된 노트북",
        "",
        f"`notebooks/{EXPERIMENT_ID}.ipynb`",
        "",
        "## 상태",
        "",
        "완료",
        "",
        "## 목적",
        "",
        "005 expanding checkpoint ensemble과 007 타입별 class-weight 모델을 pooled Platt 보정 후 soft voting으로 결합하고, alpha 선택을 walk-forward에서만 수행한 뒤 고정된 alpha를 최종 retrospective Test에 한 번만 적용한다.",
        "",
        "## 설정",
        "",
        "- Base models: 005 expanding checkpoint ensemble, 007 single type-expert + scale_pos_weight",
        f"- Alpha grid (005 weight): {ALPHAS}",
        "- Walk-forward folds: 004/005와 동일한 expanding 3-fold",
        "- Within each fold calibration and final 70~80% validation: earlier timestamp-group half for Platt fit, later half for threshold selection",
        "- Threshold rule: existing exact `select_threshold`, Recall >= 99% then max FCR",
        f"- Source integrity: dataset sha256 `{DATA_SHA256_BEFORE}`, mapping sha256 `{MAPPING_SHA256_BEFORE}`",
        f"- Log: `docs/peace/{LOG_PATH.name}`",
        "",
        "## 전체 분할",
        "",
        split_summary.reset_index().to_markdown(index=False),
        "",
        "## Walk-forward 분할",
        "",
        walk_forward_split_summary.reset_index().to_markdown(index=False),
        "",
        "## Walk-forward Calibration 내부 시간 분할",
        "",
        walk_temporal_split_summary.to_markdown(index=False),
        "",
        "## Platt 보정 계수",
        "",
        md_table(
            platt_report,
            {
                "coefficient": lambda v: f"{v:.8f}",
                "intercept": lambda v: f"{v:.8f}",
                "raw_probability_min": lambda v: f"{v:.6f}",
                "raw_probability_max": lambda v: f"{v:.6f}",
            },
        ),
        "",
        "## Fold별 Alpha 결과",
        "",
        md_table(
            walk_alpha_report,
            {
                "alpha": lambda v: f"{v:.2f}",
                "weight_005": lambda v: f"{v:.2f}",
                "weight_007": lambda v: f"{v:.2f}",
                "threshold_selected": lambda v: f"{v:.6f}",
                "threshold_recall": pct,
                "threshold_fcr": pct,
                "future_recall": pct,
                "future_fcr": pct,
                "future_pr_auc": lambda v: f"{v:.6f}",
            },
        ),
        "",
        "## Walk-forward Alpha 요약",
        "",
        md_table(
            walk_alpha_summary,
            {
                "alpha": lambda v: f"{v:.2f}",
                "mean_future_recall": pct,
                "min_future_recall": pct,
                "mean_future_fp": lambda v: f"{v:,.1f}",
                "mean_future_fcr": pct,
                "min_future_fcr": pct,
                "mean_future_pr_auc": lambda v: f"{v:.6f}",
            },
        ),
        "",
        "## Alpha 선택 규칙",
        "",
        selected_text,
        "",
        f"선택된 alpha: `{selected_alpha:.2f}`",
        "",
        "## Final Validation 내부 시간 분할",
        "",
        final_temporal_split_summary.to_markdown(index=False),
        "",
        "## Final Retrospective Test 결과",
        "",
        "아래 Test는 이전 실험들과 같은 80~100% 구간이며, 이번 retrospective에서도 alpha/threshold 선택에는 사용하지 않았다.",
        "",
        md_table(
            final_results,
            {
                "alpha": lambda v: f"{v:.2f}",
                "weight_005": lambda v: f"{v:.2f}",
                "weight_007": lambda v: f"{v:.2f}",
                "validation_threshold": lambda v: f"{v:.6f}",
                "validation_threshold_recall": pct,
                "validation_threshold_fcr": pct,
                "test_recall": pct,
                "test_fcr": pct,
                "test_pr_auc": lambda v: f"{v:.6f}",
            },
        ),
        "",
        "## 선택 Alpha와 Endpoint 비교",
        "",
        md_table(
            selected_endpoints_df[["label", "validation_threshold", "test_recall", "test_fp", "test_fcr", "test_fn", "test_tp", "test_pr_auc", "role"]],
            {
                "validation_threshold": lambda v: f"{v:.6f}",
                "test_recall": pct,
                "test_fcr": pct,
                "test_pr_auc": lambda v: f"{v:.6f}",
            },
        ),
        "",
        "## 결론",
        "",
        *conclusion_lines,
        "",
        "## 실행 로그",
        "",
        f"`docs/peace/{LOG_PATH.name}`",
    ]
)
REPORT_PATH.write_text(report_text + "\n", encoding="utf-8")
display(Markdown(report_text[:4000]))
print(f"report saved to: {REPORT_PATH.relative_to(REPO_ROOT)}")
logger.info("report_saved=%s", REPORT_PATH.name)
logger.info("experiment_complete=%s", EXPERIMENT_ID)
for handler in logger.handlers:
    handler.flush()


2026-08-25 14:56:00,127 | INFO | experiment=0825_peace_013_type_expert_fold_class_soft_voting


2026-08-25 14:56:00,128 | INFO | data_file=dataset.csv sha256=53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62


2026-08-25 14:56:00,128 | INFO | mapping_file=mapping.json sha256=3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486


2026-08-25 14:56:00,128 | INFO | versions python=3.12.7 pandas=2.2.2 sklearn=1.5.1 xgboost=3.4.1


2026-08-25 14:56:00,129 | INFO | alphas=[0.0, 0.25, 0.5, 0.75, 1.0] min_recall=0.99


...

dataset_sha256_unchanged                                                         True
mapping_sha256_unchanged                                                         True
dataset_sha256                      53e8568743216d556856ed69b388f6750fbfa0b8c59ad3...
mapping_sha256                      3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a...
selected_alpha_from_walk_forward                                                  1.0
walk_fold_count                                                                     3
walk_candidate_count                                                                5
final_candidate_count                                                               5
log_file                            docs/peace/0825_peace_013_type_expert_fold_cla...
report_file                         docs/experiments/0825_peace_013_type_expert_fo...
Name: verification, dtype: object

2026-08-25 14:56:40,967 | INFO | source_integrity=PASS


# 0825_peace_013_type_expert_fold_class_soft_voting

## 연결된 노트북

`notebooks/0825_peace_013_type_expert_fold_class_soft_voting.ipynb`

## 상태

완료

## 목적

005 expanding checkpoint ensemble과 007 타입별 class-weight 모델을 pooled Platt 보정 후 soft voting으로 결합하고, alpha 선택을 walk-forward에서만 수행한 뒤 고정된 alpha를 최종 retrospective Test에 한 번만 적용한다.

## 설정

- Base models: 005 expanding checkpoint ensemble, 007 single type-expert + scale_pos_weight
- Alpha grid (005 weight): [0.0, 0.25, 0.5, 0.75, 1.0]
- Walk-forward folds: 004/005와 동일한 expanding 3-fold
- Within each fold calibration and final 70~80% validation: earlier timestamp-group half for Platt fit, later half for threshold selection
- Threshold rule: existing exact `select_threshold`, Recall >= 99% then max FCR
- Source integrity: dataset sha256 `53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62`, mapping sha256 `3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486`
- Log: `docs/peace/0825_peace_013_type_expert_fold_class_soft_voting.log`

## 전체 분할

| split      |   rows |   positive_samples |   positive_rate_pct |   timestamp_groups | start_time                | end_time                  |
|:-----------|-------:|-------------------:|--------------------:|-------------------:|:--------------------------|:--------------------------|
| train      | 308196 |               1940 |            0.62947  |              29249 | 1970-06-23 03:58:55+00:00 | 1970-10-05 00:29:59+00:00 |
| validation |  44026 |                357 |            0.810884 |               3400 | 1970-10-05 00:30:30+00:00 | 1970-10-13 16:54:14+00:00 |
| test       |  88052 |               2325 |            2.64049  |               7093 | 1970-10-13 16:54:52+00:00 | 1970-11-02 14:21:28+00:00 |

## Walk-forward 분할

| fold   | segment     |   rows |   positive_samples |   positive_rate_pct |   timestamp_groups | start_time                | end_time                  |
|:-------|:------------|-------:|-------------------:|--------------------:|-------------------:|:--------------------------|:--------------------------|
| fold_1 | train       | 132137 |               1223 |           0.925555  |              15230 | 1970-06-23 03:58:55+00:00 | 1970-08-18 06:51:10+00:00 |
| fold_1 | calibration |  43979 |                200 |           0.454763  |               1251 | 1970-08-18 06:51:41+00:00 | 1970-08-21 23:32:59+00:00 |
| fold_1 | evaluation  |  44040 |                326 |           0.740236  |               5415 | 1970-08-21 23:33:55+00:00 | 1970-09-15 06:46:33+00:00 |
| fold_2 | train       | 176116 |               1423 |           0.80799   |              16481 | 1970-06-23 03:58:55+00:00 | 1970-08-21 23:32:59+00:00 |
| fold_2 | calibration |  44040 |                326 |           0.740236  |               5415 | 1970-08-21 23:33:55+00:00 | 1970-09-15 06:46:33+00:00 |
| fold_2 | evaluation  |  44187 |                152 |           0.343993  |               4167 | 1970-09-15 06:47:13+00:00 | 1970-09-28 05:10:37+00:00 |
| fold_3 | train       | 220156 |               1749 |           0.794437  |              21896 | 1970-06-23 03:58:55+00:00 | 1970-09-15 06:46:33+00:00 |
| fold_3 | calibration |  44187 |                152 |           0.343993  |               4167 | 1970-09-15 06:47:13+00:00 | 1970-09-28 05:10:37+00:00 |
| fold_3 | evaluation  |  43853 |                 39 |           0.0889335 |               3186 | 1970-09-28 05:11:13+00:00 | 1970-10-05 00:29:59+00:00 |

## Walk-forward Calibration 내부 시간 분할

| stage   | segment             |   rows |   positive_samples |   positive_rate_pct |   timestamp_groups | start_time                | end_time                  |
|:--------|:--------------------|-------:|-------------------:|--------------------:|-------------------:|:--------------------------|:--------------------------|
| fold_1  | platt_fit           |  21967 |                 32 |            0.145673 |                614 | 1970-08-18 06:51:41+00:00 | 1970-08-19 05:51:18+00:00 |
| fold_1  | th

report saved to: docs/experiments/0825_peace_013_type_expert_fold_class_soft_voting.md
2026-08-25 14:56:40,979 | INFO | report_saved=0825_peace_013_type_expert_fold_class_soft_voting.md


2026-08-25 14:56:40,980 | INFO | experiment_complete=0825_peace_013_type_expert_fold_class_soft_voting
